# ALMs — Training Authorial Language Models (Ch.5)

This notebook walks through **stage 1 of the ALMs method** (Ch.5 Sec 5.3.1 of
Huang's thesis): *further-pretraining* one GPT-2 per candidate author on that
author's known writings. Each resulting model is an **Authorial Language
Model (ALM)**. A later notebook (`ALMs_PPL.ipynb`) will score questioned
documents under each ALM and attribute by lowest perplexity.

**The idea in one paragraph.** GPT-2 is already fluent in English; further
pretraining nudges its parameters so it becomes *especially good* at
predicting one author's word sequences. If we later see an anonymous text,
the author whose model finds it most predictable (lowest perplexity) is our
best guess for who wrote it.

**Why one model per author?** A single LLM cannot capture many distinct
styles at once — the thesis (Ch.5 Sec 5.2) argues this is why the earlier
"pALM" approach (one shared model with per-author heads) underperformed.
ALMs devotes each model's full capacity to a single author's style.

**Debug vs. thesis configuration.** The thesis recipe (Ch.5 Table 2) is
100 epochs, gradient-accumulation 64, block size 128 on `gpt2` — hours per
author on an A100. This demo uses the shipped natural demo corpus with reduced
hyperparameters (`epochs=15`, `batch_size=1`, `block_size=64`):
roughly 10 minutes per author on a GPU-class device (xpu), and a
few minutes more on CPU. The mapping back to the thesis
configuration is shown at the end.

## 0. Bootstrap

The `thesis_aa` package is not pip-installed (there is no `pyproject.toml`),
so we insert the repository root into `sys.path` first — the same trick
`tests/conftest.py` uses. After this cell, `import thesis_aa` works no
matter where the notebook kernel was started.

In [1]:
import os, sys

REPO_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))  # notebooks/ -> repo root
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from thesis_aa import config, data as data_mod
from thesis_aa.alms import train as alms_train

print('repo root :', REPO_ROOT)
print('device    :', config.get_device())
print('base model:', config.DEFAULT_BASE_MODEL)

C:\Users\MiraMoe\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+xpu).


W0904 13:52:16.530000 968 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


repo root : D:\AgentHome\Thesis
device    : xpu
base model: gpt2


**What you should see:** the repository root, the auto-detected torch
device (`cuda` → `xpu` → `cpu`, whichever is available), and the default
base model (`gpt2`, the 124M-parameter variant — the smallest of the four
sizes in `config.MODEL_SIZES`).

## 1. Get a corpus

Every experiment needs a dataframe of ``(text, author_tag)`` rows. We use
the **natural demo corpus** shipped with the repository (`data/natural/`):
five author personas writing in distinct topical domains — castle life,
ocean science, cookery, law, polar travel. We further-pretrain on the
castle / ocean / law personas (`author00`–`author02`), 50 documents each,
so the demo's 15-epoch training costs ~10 minutes per author on xpu. (The pytest suite instead uses the
instant `generate_synthetic` corpus — pipeline mechanics don't care about
prose quality.)


In [2]:
# Three-author subset of the shipped natural demo corpus.
full_train, full_test = data_mod.load_natural()
keep = {'author00', 'author01', 'author02'}
train_df = full_train[full_train['author_tag'].isin(keep)].reset_index(drop=True)
test_df = full_test[full_test['author_tag'].isin(keep)].reset_index(drop=True)

print('train shape:', train_df.shape, '| test shape:', test_df.shape)
print('columns     :', list(train_df.columns))
print()
print(train_df['author_tag'].value_counts())


train shape: (150, 2) | test shape: (60, 2)
columns     : ['text', 'author_tag']
author_tag
author00    50
author01    50
author02    50
Name: count, dtype: int64


**What you should see:** 150 training rows (3 authors × 50 documents) of
ordinary English prose, plus 60 test rows held out. The personas share
most of the core vocabulary but diverge in their topical domains and
stylistic habits — the signal the authorial models latch onto within
a few epochs of further pretraining.


In [3]:
for tag, group in train_df.groupby('author_tag'):
    print(f'--- {tag} (first 110 chars) ---')
    print(group['text'].iloc[0][:110])

--- author00 (first 110 chars) ---
<BOS>The falconer released his bird at sunrise, and the whole household paused to watch it climb above the tow
--- author01 (first 110 chars) ---
<BOS>An isotope ratio from the mass spectrometer confirmed the upwelling hypothesis, and the chief scientist r
--- author02 (first 110 chars) ---
<BOS>The court's list was rearranged to accommodate the long-running trial, and the fixture stood for eight we


Each author's domain vocabulary is clearly distinct — castle, ocean,
courtroom — wrapped in ordinary English. That is exactly the setting in
which one model per author can specialise: the further-pretrained ALM
nudges GPT-2's expectations toward *this* author's word sequences.


## 2. What happens to the text before training?

Before fine-tuning, `alms.train` turns each author's documents into the
standard causal-LM training format:

1. **Tokenize** each document with the GPT-2 tokenizer.
2. **Concatenate** all the author's token ids into one long stream.
3. **Chunk** the stream into fixed `block_size` blocks — each block is one
   training example.

The chunking is exactly the classic `group_texts` function from the
HuggingFace language-modeling examples. Let's run it on one author's corpus
to see the mechanics — this is the step that most "how does LM training
data look?" questions are about.

In [4]:
from datasets import Dataset
from transformers import AutoTokenizer

# The internals used by train_authorial_model (exposed here for the demo):
one_author = train_df[train_df['author_tag'] == 'author00'].reset_index(drop=True)
ds = Dataset.from_pandas(one_author[['text']])
print('author00 documents:', len(ds))

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenized = ds.map(alms_train._preprocess, fn_kwargs={'tokenizer': tokenizer},
                   batched=True, remove_columns=ds.column_names)
stream_len = sum(len(ids) for ids in tokenized['input_ids'])
print('concatenated token stream:', stream_len, 'tokens')

block_size = 64  # demo value; thesis uses 128
chunked = tokenized.map(alms_train._group_texts, fn_kwargs={'block_size': block_size}, batched=True)
print('training blocks:', len(chunked),
      f'(each {block_size} tokens; a tail shorter than block_size is dropped)')
print('first block starts with token ids:', chunked[0]['input_ids'][:8], '...')

author00 documents: 50


concatenated token stream: 14386 tokens


training blocks: 224 (each 64 tokens; a tail shorter than block_size is dropped)
first block starts with token ids: [27, 33, 2640, 29, 464, 24215, 1102, 263] ...


**What you should see:** the author's 50 documents become one token
stream, which is chopped into dozens of 64-token blocks. Each block
carries an identical `labels` copy (GPT-2's Trainer shifts them internally
— position *i* is trained to predict token *i+1*). With `block_size=128`
and real corpora, the same code produces hundreds of blocks per author.

## 3. Train one ALM per author

`train_all_authors` loops over the unique `author_tag` values and calls
`train_authorial_model` for each. Per author it:

- splits the author's rows 80/20 (the 20% is a *held-out eval split* used
  only to report a perplexity — it is not the attribution test set),
- tokenizes + chunks as above,
- further-pretrains `gpt2` for `epochs` epochs with the HuggingFace
  `Trainer`,
- saves model + tokenizer under `models/<author_tag>/`,
- appends the tag to `log/done.txt` (the resumability log).

**Debug hyperparameters below.** The thesis values (Ch.5 Table 2) are
`epochs=100, gradient_accumulation_steps=64, block_size=128` — see
`config.ALMS_TRAIN_CONFIG`. We pass the CPU-friendly equivalents instead.
Note `fp16` auto-enables only under CUDA; on this machine's `xpu` build the
run is fp32, which is fine for a demo.

In [5]:
results = alms_train.train_all_authors(
    train_df,
    base_model='gpt2',   # thesis also evaluates larger sizes via config.MODEL_SIZES
    out_dir=config.MODEL_DIR,
    log_home=config.LOG_DIR,
    epochs=15,                       # thesis: 100 (config.ALMS_TRAIN_CONFIG['epochs'])
    gradient_accumulation_steps=1,  # thesis: 64
    batch_size=1,
    block_size=64,                  # thesis: 128
    fp16=False,
)
print('Saved models:', list(results))

[ALMs] Fetched 0 tasks: []
Saved models: []


**What you should see:** a progress bar per author (15 epochs each) and
a final `[ALMs] authorNN: Perplexity: X.XX` line — the held-out
perplexity of each author's model on its own 20% eval split (10 held-out
documents). Fifteen epochs move GPT-2 far enough on the natural corpus; the thesis's attribution power comes from 100
further-pretraining epochs (Ch.5 Table 5.1) — raise `epochs` and re-run
for the real behaviour.


## 4. Inspect the artifacts

Training writes four kinds of artifacts. Let's look at each.

In [6]:
import os

print('=== models/ directory layout ===')
for tag in sorted(os.listdir(config.MODEL_DIR)):
    path = os.path.join(config.MODEL_DIR, tag)
    if os.path.isdir(path):
        files = sorted(os.listdir(path))
        print(f'  {tag}/ -> {files}')

print()
print('=== per-author held-out perplexity (eval.txt) ===')
for tag in sorted(os.listdir(config.MODEL_DIR)):
    eval_path = os.path.join(config.MODEL_DIR, tag, 'eval.txt')
    if os.path.isfile(eval_path):
        print(f'  {tag}: {open(eval_path).read().strip()}')

=== models/ directory layout ===
  author00/ -> ['config.json', 'eval.txt', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.json']
  author01/ -> ['config.json', 'eval.txt', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.json']
  author02/ -> ['config.json', 'eval.txt', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.json']
=== per-author held-out perplexity (eval.txt) ===
  author00: Perplexity: 313.81
  author01: Perplexity: 254.72
  author02: Perplexity: 174.99


`models/<tag>/` is a standard HuggingFace checkpoint directory
(`model.safetensors` + tokenizer files + `config.json`). Its presence is
also the "already trained?" check — retraining an author whose directory
exists is skipped. `eval.txt` accumulates one perplexity line per completed
training run for that author.

In [7]:
print('=== resumability logs (log/) ===')
print('done.txt :', open(os.path.join(config.LOG_DIR, 'done.txt')).read().split())

=== resumability logs (log/) ===
done.txt : ['author00', 'author01', 'author02']


`log/done.txt` lists every author that finished training. If the run
is interrupted (say, after author00 on a 50-author benchmark), restarting
`train_all_authors` reads this file and skips the completed tags — a
crash halfway through a multi-hour run costs only the unfinished authors.
`log/target.txt` (optional) restricts training to a whitelist of tags.

### Resumability in action

Call `train_all_authors` again — nothing retrains:

In [8]:
results2 = alms_train.train_all_authors(
    train_df, epochs=15, gradient_accumulation_steps=1,
    batch_size=1, block_size=64, fp16=False,
)
print('re-run returned (nothing retrained):', list(results2))

[ALMs] Fetched 0 tasks: []
re-run returned (nothing retrained): []


**What you should see:** `[ALMs] Fetched 0 tasks: []` — all three
authors are in `done.txt`, so the second call is a no-op. (Note the
skip-check in `train_authorial_model` is on `models/<tag>/config.json`
existing, so it protects even if `log/` is deleted.)

## 5. Going to real data

To reproduce the thesis configuration (Ch.5 Table 2):

| Parameter | Demo (this notebook) | Thesis |
|---|---|---|
| Corpus | 3-author synthetic | Blogs50 / CCAT50 / Guardian / IMDB62 |
| Base model | `gpt2` (124M) | `gpt2` (+ medium/large/xl in ablations) |
| Epochs | 15 | 100 |
| Gradient accumulation | 1 | 64 |
| Block size | 64 | 128 |

```python
# Real data + thesis hyperparameters:
train_df, test_df = data_mod.load_benchmark('Blogs50')      # or download_benchmark_subset
results = alms_train.train_all_authors(
    train_df,
    epochs=config.ALMS_TRAIN_CONFIG['epochs'],               # 100
    gradient_accumulation_steps=config.ALMS_TRAIN_CONFIG['gradient_accumulation_steps'],  # 64
    block_size=config.ALMS_TRAIN_CONFIG['block_size'],      # 128
    learning_rate=config.ALMS_TRAIN_CONFIG['learning_rate'], # 2e-5
)
```

On a single A100 the thesis configuration costs roughly *hours per author ×
50 authors* — the resumability logs (`log/done.txt`) exist precisely for
runs of this scale. Once the models exist, continue to
**`ALMs_PPL.ipynb`** to score questioned documents and benchmark the
attribution accuracy.